# エージェント型動画分析

このノートブックでは、2つの異なるアプローチを使用して動画分析用のAIエージェントを構築する方法を実証します：

1. **TwelveLabs API**: インデックス作成とセマンティック検索による高度な動画理解
2. **AWS Bedrock Pegasus**: Amazonのマルチモーダルモデルを使用した直接動画分析

## 学習内容

- 専門化された動画分析エージェントを作成する
- AI分析のために動画をアップロードして処理する
- 自然言語で動画コンテンツをクエリする
- 異なる動画分析アプローチを比較する
- ローカルファイルとクラウドストレージの両方を処理する

## 前提条件

**TwelveLabs APIの場合:**
- [TwelveLabs Platform](https://playground.twelvelabs.io/)からのTwelveLabs APIキー
- Python 3.9+環境

**Amazon Bedrockの場合:**
- [Amazon Bedrockモデルアクセス](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)権を持つAWSアカウント
- 適切な権限で設定された[AWS CLI](https://aws.amazon.com/cli/)
- 動画ストレージ用の[Amazon S3バケット](https://aws.amazon.com/s3/)

## セットアップ

まず、必要なライブラリをインポートします：

In [ ]:
import os
from strands import Agent


## Option 1: TwelveLabs Video Analysis

TwelveLabs provides advanced video understanding capabilities with indexing, semantic search, and detailed video insights.

In [ ]:
# Install TwelveLabs SDK
!pip install twelvelabs

In [ ]:
# Configure TwelveLabs API key
# Get your API key from: https://playground.twelvelabs.io/dashboard/api-key
os.environ['TL_API_KEY'] = "your-twelvelabs-api-key"


In [ ]:
# Create TwelveLabs video analysis agent
from twelvelabs_video_tool import twelvelabs_video_analysis

twelvelabs_agent = Agent(
    tools=[twelvelabs_video_analysis],
    system_prompt="""You are a specialized video analysis agent using TwelveLabs API. You can:
    
    1. Upload videos to create searchable indexes
    2. Generate video insights (titles, topics, hashtags)
    3. Answer detailed questions about video content
    4. List and search through video collections
    
    Always be helpful and provide comprehensive video analysis.
    When users ask about videos, first check what videos are available.
    """
)


### Optional: Use Anthropic Claude as Model provider


In [ ]:
# Install Anthropic integration
!pip install 'strands-agents[anthropic]'

In [ ]:
from strands.models.anthropic import AnthropicModel

# Configure Anthropic API key
# Get your API key from: https://console.anthropic.com/
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY") or "your-anthropic-api-key"


anthropic_model = AnthropicModel(
    client_args={"api_key": ANTHROPIC_API_KEY},
    model_id="claude-3-7-sonnet-20250219", #or us.claude-3-7-sonnet-20250219 ,see https://docs.aws.amazon.com/bedrock/latest/userguide/models-supported.html 
    max_tokens=1024,
    params={"temperature": 0.3}
)

# Update agent to use Anthropic model
twelvelabs_agent.model = anthropic_model


### Example 1: List Available Videos

Let's see what videos are available in your TwelveLabs indexes:

In [ ]:
# List all available videos across all indexes
response = twelvelabs_agent("List all available videos")
print(response)

### Example 2: Analyze Existing Video

If you have videos in your indexes, you can ask questions about them:

In [ ]:
# Summarize an existing video (replace with actual video ID from the list above)
response = twelvelabs_agent("When is S3 mentioned in the video?")
print(response)

### Example 3: Upload and Analyze New Video

Upload a new video file for analysis:

In [ ]:
# Upload and analyze a new video
# Replace with your actual video file path
video_path = "/path/to/your/video.mp4"

response = twelvelabs_agent(f"Upload and analyze the video at {video_path} with the name 'sample-video'")
print(response)

## Option 2: AWS Bedrock Video Analysis

AWS Bedrock with Pegasus model provides direct video analysis without requiring video indexing.

In [ ]:
# Configure AWS settings
# Make sure your AWS credentials are configured via AWS CLI or environment variables
os.environ['S3_BUCKET_NAME'] = 'your-video-bucket'  # Replace with your S3 bucket
os.environ['AWS_REGION'] = 'us-east-1'  # Replace with your preferred region

print("🔧 AWS Configuration:")
print(f"AWS Region: {os.environ.get('AWS_REGION')}")
print(f"S3 Bucket: {os.environ.get('S3_BUCKET_NAME')}")

In [ ]:
# Create AWS Bedrock video analysis agent
from bedrock_video_tool import bedrock_video_analysis

bedrock_agent = Agent(
    tools=[bedrock_video_analysis],
    system_prompt="""You are a specialized video analysis agent using AWS Bedrock Pegasus model. You can:
    
    1. Analyze videos directly from local files or S3 URIs
    2. Automatically upload local videos to S3 when needed
    3. Answer questions about video content and scenes
    4. List and search videos in S3 buckets
    
    Always be helpful and provide detailed video analysis.
    When analyzing local files, I'll automatically upload them to S3 first.
    """
)

print("✅ AWS Bedrock agent created!")

### Example 4: List Videos in S3 Bucket

In [ ]:
# List available videos in S3 bucket
response = bedrock_agent("List all available videos in the S3 bucket")
print(response)

In [ ]:
response = bedrock_agent(f"Analyze the video2 and tell me what it's about")

In [ ]:
response = bedrock_agent("When is S3 mentioned in the video?")

### Example 5: Analyze Video with Bedrock

Analyze a video using AWS Bedrock Pegasus model:

In [ ]:
# Analyze a video file (will be automatically uploaded to S3 if local)
# Replace with your actual video file path
video_path = "/path/to/your/video.mp4"

response = bedrock_agent(f"Analyze the video '{video_path}' and tell me what it's about")
print(response)

## Comparison: TwelveLabs vs AWS Bedrock

| Feature | TwelveLabs | Amazon Bedrock |
|---------|------------|-------------|
| **Setup** | API key required | AWS credentials required |
| **Video Processing** | Index-based, persistent | Direct analysis |
| **Video Insights** | Rich metadata (titles, topics, hashtags) | Content-focused analysis |
| **Storage** | TwelveLabs cloud | Your Amazon S3 bucket |
| **Cost Model** | Per API call | Per analysis request |

## Troubleshooting

**Common Issues:**

1. **API Key Errors**: Ensure your TwelveLabs API key is valid and has sufficient credits
2. **AWS Permissions**: Verify your AWS credentials have Bedrock and S3 access
3. **Video Format**: Both services support common formats (MP4, MOV, AVI, etc.)
4. **File Size**: Check service limits for maximum video file sizes

For more help, refer to:
- [TwelveLabs Documentation](https://docs.twelvelabs.io/)
- [Amazon Bedrock Documentation](https://docs.aws.amazon.com/bedrock/)
- [Strands Agents Documentation](https://strandsagents.com/)